# TrustLens Day 2 — Deepfake Classifier (EfficientNet-B0) on Colab

Transfer-learn EfficientNet-B0 on **GenImage**, holding out **one generator family** as out-of-distribution (OOD). Headline result: the **OOD accuracy drop**.

**Runtime → Change runtime type → GPU (T4).** Budget ~2–3 h.
Uses the disk-light **GenImage Validation** subset (all generators, small).

> Run top-to-bottom. The only manual step is uploading `kaggle.json` in cell 2.

## 0. GPU check

In [ ]:
!nvidia-smi -L
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Get the TrustLens code + install

Clones the repo (must contain the **Day-2** code — push it first if you haven't: `git add -A && git commit -m 'Day 2' && git push`).

In [ ]:
![ -d TrustLens ] && (cd TrustLens && git pull -q) || git clone -q https://github.com/umerjavaidkh/TrustLens.git
%cd TrustLens
!pip -q install mlflow pyngrok kaggle
!pip -q install -e .
import sys; sys.path.insert(0, "src")
import trustlens.training.train, trustlens.models.splits
print("TrustLens Day-2 modules loaded OK")

## 2. Get GenImage data — Validation subset (disk-light)

Full per-generator subsets are ~90 GB each — too big for Colab. The Validation subset bundles all generators and fits on disk.

**First:** Kaggle → Settings → API → *Create New Token* → downloads `kaggle.json`.

In [ ]:
# Clear any partial download, then upload kaggle.json when prompted
!rm -rf /content/genimage
from google.colab import files
files.upload()                       # <-- select kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!df -h /content | tail -1

In [ ]:
import os
os.makedirs("/content/genimage", exist_ok=True)
!kaggle datasets download -d vtphatt2/GenImage-Validation -p /content/genimage --unzip
print("--- structure (dirs) ---")
!find /content/genimage -maxdepth 3 -type d | head -60
!df -h /content | tail -1

## 3. Auto-detect data root + generators + OOD family
No manual path typing — finds the folder level whose children are the generators, and picks the BigGAN family as OOD (falls back to the first generator).

In [ ]:
from pathlib import Path
from trustlens.models.splits import index_dataset, list_generators

NOISE = {"val", "train", "ai", "nature", "real", "fake"}

def autodetect_root(base):
    base = Path(base)
    tries = [base, base / "val", base / "train"]
    tries += [p for p in base.iterdir() if p.is_dir()]
    best, best_gens, seen = str(base), [], set()
    for t in tries:
        t = str(t)
        if t in seen or not Path(t).exists():
            continue
        seen.add(t)
        gens = [g for g in list_generators(index_dataset(t)) if g.lower() not in NOISE]
        if len(gens) > len(best_gens):
            best, best_gens = t, gens
    return best, best_gens

DATA_ROOT, GENERATORS = autodetect_root("/content/genimage")
print("DATA_ROOT   =", DATA_ROOT)
print("GENERATORS  =", GENERATORS)

def pick_ood(gens, prefer="biggan"):
    for g in gens:
        if prefer in g.lower():
            return g
    return gens[0]

OOD_GEN = pick_ood(GENERATORS)
print("OOD_GENERATOR =", OOD_GEN, " (edit manually to hold out a different family)")
assert len(GENERATORS) >= 2, 'Need >=2 generators for an OOD split; check structure above.'

## 4. Config

In [ ]:
from trustlens.training.train import TrainConfig
cfg = TrainConfig(
    data_root=DATA_ROOT,
    ood_generator=OOD_GEN,
    img_size=224,
    batch_size=64,
    head_epochs=3,                # Phase 1: frozen backbone
    finetune_epochs=5,            # Phase 2: unfreeze last blocks
    unfreeze_blocks=2,
    max_per_class_per_gen=2000,   # cap to fit ~2-3h; raise if you have GPU time
    target_fpr=0.01,
    experiment="trustlens-deepfake",
    out_dir="models",
)
cfg

## 5. Train (two-phase) + evaluate ID vs OOD

In [ ]:
from trustlens.training.train import train
model, report, logs = train(cfg)

## 6. The distribution-shift exhibit

In [ ]:
import json, matplotlib.pyplot as plt
print(report.summary_line())
print(json.dumps(report.as_dict(), indent=2))

labels = ["ID test", f"OOD\n({report.ood_generator})"]
accs = [report.id_test.accuracy, report.ood.accuracy]
plt.bar(labels, accs, color=["#2a9d8f", "#e76f51"]); plt.ylim(0, 1)
plt.ylabel("accuracy")
plt.title(f"OOD accuracy drop = {report.accuracy_drop:.3f} ({report.relative_drop:.0%})")
for i, a in enumerate(accs): plt.text(i, a + 0.01, f"{a:.3f}", ha="center")
plt.show()

## 7. Inspect the MLflow run

In [ ]:
import mlflow
run = mlflow.search_runs(experiment_names=[cfg.experiment]).iloc[0]
print(run[[c for c in run.index if c.startswith("metrics.")]])
# Live UI (needs ngrok token):
# from pyngrok import ngrok
# get_ipython().system_raw('mlflow ui --port 5000 &'); print(ngrok.connect(5000))

## 8. Save the best model

In [ ]:
print("Best checkpoint:", "models/efficientnet_b0_best.pt")
# from google.colab import drive; drive.mount('/content/drive')
# import shutil; shutil.copy('models/efficientnet_b0_best.pt',
#     '/content/drive/MyDrive/trustlens_efficientnet_b0_best.pt')

## Notes

- **Why hold a family out?** Fraudsters use *new* generators; in-distribution accuracy overstates real performance. The OOD drop is the number that matters.
- **Time budget:** `max_per_class_per_gen` caps images per (generator, class). Start 2000; raise it or add epochs if you have GPU time.
- **Different hold-out:** set `OOD_GEN` (cell 3) to any name in `GENERATORS`.